### Preprocessing step 3: Feature Selection

In [ ]:
import pandas as pd
import os
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import numpy as np
from tqdm import tqdm
from helperfunctions.intern_constants import TS_COL, WT_ID
from helperfunctions import intern_constants as ic


In [ ]:
def check_duplicates(df_r_turbine):
    df_r_turbine[TS_COL] = pd.to_datetime(df_r_turbine[TS_COL], errors="coerce")

    duplicates = df_r_turbine[TS_COL].duplicated()

    num_dupl = duplicates.sum()

    if num_dupl > 0:
        print("Duplicated timestamps")
        print(df_r_turbine[duplicates].head(10))

In [ ]:
def print_cols(df_r_turbine):
    df_r_turbine.columns = df_r_turbine.columns.str.strip()

    for i,it in enumerate(df_r_turbine.columns):
        print(f"{i:2d} {it}")

In [ ]:
columns_to_select = [
    'Ambient temperature (converter) (°C)',
    'Date and time',
    'Drive train acceleration (mm/ss)',
    'Gear oil inlet pressure (bar)',
    'Gear oil pump pressure (bar)',
    'Gearbox speed (RPM)',
    'Generator bearing front temperature (°C)',
    'Generator bearing rear temperature (°C)',
    'Generator RPM (RPM)',
    'Hub temperature (°C)',
    'Motor current axis 1 (A)',
    'Motor current axis 2 (A)',
    'Motor current axis 3 (A)',
    'Nacelle ambient temperature (°C)',
    'Nacelle position (°)',
    'Nacelle temperature (°C)',
    'Power (kW)',
    'Rotor bearing temp (°C)',
    'Rotor speed (RPM)',
    'Stator temperature 1 (°C)',
    'Temp. top box (°C)',
    'Temperature motor axis 1 (°C)',
    'Temperature motor axis 2 (°C)',
    'Temperature motor axis 3 (°C)',
    'Vane position 1+2 (°)',
    'Wind direction (°)',
    'Wind speed (m/s)',
    'WT_ID',
    "Blade angle (pitch position) A (°)",
    "Blade angle (pitch position) B (°)",
    "Blade angle (pitch position) C (°)",
    "Front bearing temperature (°C)",
    "Gear oil inlet temperature (°C)",
    "Gear oil temperature (°C)",
    "Rear bearing temperature (°C)",
    "Tower Acceleration X (mm/ss)",
    "Tower Acceleration y (mm/ss)",
    "Transformer cell temperature (°C)",
    "Transformer temperature (°C)",
    "Yaw bearing angle (°)",
]

### Read summarized files and filter signals

In [ ]:
path = Path(ic.PATH_PROJECT_ROOT / "src" / "Penmanshiel")
folder_path = os.path.join(path, "processed_data")
os.makedirs(folder_path, exist_ok=True)
# WT3 does not exist
WT_IDs = [1,2,4,5,6,7,8,9,10,11,12,13,14,15]

# casing error
correct_col = {"Tower Acceleration y (mm/ss)": "Tower Acceleration Y (mm/ss)"}

for wt_ID in tqdm(WT_IDs):
    path_to_file = os.path.join(path,f"WT_Data_Complete_ID_{wt_ID}.csv")
    df_r_turbine = pd.read_csv(path_to_file)

    check_duplicates(df_r_turbine)
    #print_cols(df_r_turbine)
    df_filtered = df_r_turbine[columns_to_select]
    # rename false col
    df_filtered.rename(columns=correct_col, inplace=True)
    # Set column WT_ID at the beginning of the table
    cols = [WT_ID] + [col for col in df_filtered if col !='WT_ID']
    df_filtered = df_filtered[cols]
    
    df_filtered.to_csv(os.path.join(folder_path,f"WT_ID_{wt_ID}_selected_signals.csv"),index=False)
    
    try:
        os.remove(path_to_file)
    except FileNotFoundError:
        pass

### Check

In [ ]:
wt_ID = 1
path_to_file = os.path.join(path/"processed_data",f"WT_ID_{wt_ID}_selected_signals.csv")
df = pd.read_csv(path_to_file)

#df.columns = df.str.strip()

for i,it in enumerate(df.columns):
    print(f"{i:2d} {it}")

df.head()

In [ ]:
path = Path(ic.PATH_PROJECT_ROOT / "src" / "Penmanshiel")
folder_path = os.path.join(path, "processed_data")
files = glob(os.path.join(folder_path, "*.csv"))

for f in tqdm([files[0]]):
    
    df = pd.read_csv(f)

    

    df[TS_COL] = pd.to_datetime(df[TS_COL], errors="coerce")
    df = df.set_index(TS_COL).sort_index()
    
    full_idx = pd.date_range(df.index.min(), df.index.max(), freq="10T")
    
    df = df.reindex(full_idx)    
    
    data = df.select_dtypes(include=["float", "int"])
    # Min-Max
    
    denom = (data.max() - data.min()).replace(0, np.nan)
    
    normalized = (data - data.min()) / denom
    
    plt.figure(figsize=(28,14))
    ax = sns.heatmap(normalized.T, cmap="viridis", cbar=True)

    cols = pd.to_datetime(normalized.T.columns)
    labels = cols.strftime("%Y-%m-%d")
    
    k = max(len(cols) // 12, 1)
    pos = np.arange(0, len(cols), k)
    
    ax.set_xticks(pos)
    ax.set_xticklabels(labels[pos], rotation=0, ha="right", fontsize=14)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=14)
    #plt.title("Heatmap of Selected Signals Over Time from Kelmarsh wind turbine 1")
    ax.set_xlabel("Time")
    ax.set_title(f"Unprocessed Raw Signals of WT {str(df["WT_ID"][0])}", fontsize=18)
    #ax.set_ylabel("Signals")
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=14)
    plt.tight_layout()
    plt.savefig(Path(ic.PATH_PRINTS / f"raw_sigs_overview_wt{str(df["WT_ID"][0])}"))
    plt.show()